**Initial Data Report and  Upload Data**

In [2]:
import sweetviz as sv
import pandas as pd
df = pd.read_excel("oasis_cross-sectional-5708aa0a98d82080.xlsx")

In [ ]:
report = sv.analyze(df)
report.show_html("Oasis1 data report.html")

**Data Cleaning**
Filter columns
Remove duplicates, Format Names and Columns, Change data types, Drop columns, Add columns

In [3]:
df.shape

(436, 12)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 436 entries, 0 to 435
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      436 non-null    str    
 1   M/F     436 non-null    str    
 2   Hand    436 non-null    str    
 3   Age     436 non-null    int64  
 4   Educ    235 non-null    float64
 5   SES     216 non-null    float64
 6   MMSE    235 non-null    float64
 7   CDR     235 non-null    float64
 8   eTIV    436 non-null    int64  
 9   nWBV    436 non-null    float64
 10  ASF     436 non-null    float64
 11  Delay   20 non-null     float64
dtypes: float64(7), int64(2), str(3)
memory usage: 41.0 KB


In [5]:
df.describe()

,Age,Educ,SES,MMSE,CDR,eTIV,nWBV,ASF,Delay
count,436.000000,235.000000,216.000000,235.00000,235.000000,436.000000,436.000000,436.000000,20.00000
mean,51.357798,3.178723,2.490741,27.06383,0.285106,1481.919725,0.791670,1.198894,20.55000
std,25.269862,1.311510,1.120593,3.69687,0.383405,158.740866,0.059937,0.128682,23.86249
min,18.000000,1.000000,1.000000,14.00000,0.000000,1123.000000,0.644000,0.881000,1.00000
25%,23.000000,2.000000,2.000000,26.00000,0.000000,1367.750000,0.742750,1.111750,2.75000
50%,54.000000,3.000000,2.000000,29.00000,0.000000,1475.500000,0.809000,1.190000,11.00000
75%,74.000000,4.000000,3.000000,30.00000,0.500000,1579.250000,0.842000,1.284250,30.75000
max,96.000000,5.000000,5.000000,30.00000,2.000000,1992.000000,0.893000,1.563000,89.00000


In [6]:
df.dtypes

ID           str
M/F          str
Hand         str
Age        int64
Educ     float64
SES      float64
MMSE     float64
CDR      float64
eTIV       int64
nWBV     float64
ASF      float64
Delay    float64
dtype: object

In [7]:
df.isna().sum()

ID         0
M/F        0
Hand       0
Age        0
Educ     201
SES      220
MMSE     201
CDR      201
eTIV       0
nWBV       0
ASF        0
Delay    416
dtype: int64

In [8]:
df = df.drop(columns=['Hand','ASF','Delay'])

In [9]:
df.shape

(436, 9)

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df = df[df['Age'] >= 60]

In [12]:
df.shape

(198, 9)

In [13]:
df['mr_delay'] = 0
df['dataset_source'] = 'oasis1'
df['visit'] = 1

In [14]:
df.isna().sum()

ID                 0
M/F                0
Age                0
Educ               0
SES               18
MMSE               0
CDR                0
eTIV               0
nWBV               0
mr_delay           0
dataset_source     0
visit              0
dtype: int64

In [15]:
mapping = {
    "ID" : "patient_id",
    "M/F" : "gender",
    "Age" : "age",
    "Educ" : "education",
    "SES" : "ses",
    "MMSE" : "mmse",
    "CDR" : "cdr",
    "eTIV" : "etiv",
    "nWBV" : "nwbv",
    "mr_delay" : "mr_delay",
    "dataset_source" : "dataset_source",
    "visit" : "visit"
}

df = df.rename(columns = mapping)

In [16]:
df['patient_id'] = df['patient_id'].str.strip().str.replace('_MR1','', regex=False)

In [17]:
def group(cdr):
    if cdr == 0:
        return 'Nondemented'
    elif cdr > 0:
        return 'Demented'
    else:
        return 'None' 

df['group'] = df['cdr'].apply(group)    

In [18]:
df.shape

(198, 13)

In [19]:
df['ses'] = df['ses'].fillna(df['ses'].median())

In [20]:
df['ses']  = df['ses'].astype(int)
df['education'] = df['education'].astype(int)

In [21]:
df = df.reset_index(drop=True)

In [23]:
df['group'].value_counts()

group
Demented       100
Nondemented     98
Name: count, dtype: int64

In [24]:
df = df [
    [
        "patient_id",
        "group",         
        "visit",         
        "mr_delay",      
        "gender",        
        "age",           
        "education",     
        "ses",           
        "mmse",          
        "cdr",           
        "etiv",          
        "nwbv",  
        "dataset_source"
    ]
]

In [27]:
df.isna().sum()

patient_id        0
group             0
visit             0
mr_delay          0
gender            0
age               0
education         0
ses               0
mmse              0
cdr               0
etiv              0
nwbv              0
dataset_source    0
dtype: int64

**Save Cleaned Data**

In [30]:
df.to_csv("oasis1_snapshot.csv", index=False)